# NB_Purview_Copilot_BronzeToSilver
**Source:** `AI_Medallion.Bronze.purview_copilot_usage` (OneLake deltalake scan, same as Cursor bronze)  
**Targets (reads):** `okta_user_for_ai`, `hc_silver_historical`  
**Target:** `AI_Medallion.Silver.purview_copilot_usage` (+ `purview_copilot_usage_okta_reject`)  
**Run after:** Okta + HC silver notebooks  

**Run control:** `COPILOT_CLOSE_MONTH_NO` — month-close filters CreationDate through that month and replaces that month in silver; daily uses `PURVIEW_COPILOT_MIN_CREATION_DATE`. Frozen months: `Gold.ingest_freeze_control` (`product` copilot|all).  

**Local Windows (large volume):** defaults use driver-side merge staging (`MEDALLION_STAGING_SPARK_PARQUET=0`) — same idea as Cursor, but Purview skips extra `break_lineage_local` before staging.

In [ ]:
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
REBUILD_MEDALLION_CACHE = True  # True: rebuild; False: use cache
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "75000")
os.environ.setdefault("PYSPARK_DRIVER_MEMORY", "8g")
# Large Purview silver: avoid JVM parquet OOM (driver-side chunked staging).
os.environ["MEDALLION_STAGING_SPARK_PARQUET"] = "0"
os.environ.setdefault("MEDALLION_STAGING_COLLECT_ROWS", "15000")
os.environ.setdefault("MEDALLION_STAGING_COLLECT_PARTITIONS", "8")
COPILOT_CLOSE_MONTH_NO = None
# COPILOT_CLOSE_MONTH_NO = 202609
from Common.io import sync_copilot_close_month_env, sync_rebuild_medallion_cache_env

sync_rebuild_medallion_cache_env(REBUILD_MEDALLION_CACHE)
sync_copilot_close_month_env(COPILOT_CLOSE_MONTH_NO)


In [ ]:
%run ./Common/bootstrap


In [ ]:
bronze = read_table_broken_lineage(
    TABLE_PURVIEW_COPILOT_BRONZE,
    "purview_copilot_bronze",
)


In [ ]:
okta = read_table_broken_lineage(TABLE_OKTA_SILVER, "okta_silver")
okta = filter_through_close_month_ts(
    okta, "OktaUserUpdatedAt", copilot_close_month_no()
)


In [ ]:
hc = read_table_broken_lineage(TABLE_HC_SILVER, "hc_silver")
hc = filter_to_month_no(hc, "Month_No", copilot_close_month_no())


In [ ]:
_CLOSE = copilot_close_month_no()
_FROZEN = copilot_frozen_month_nos_to_skip(spark, _CLOSE)
print(f"copilot silver: close_month={_CLOSE!r} frozen_skip={_FROZEN}", flush=True)

# Daily: [] = all bronze files. Month-close: optional filename filter.
COPILOT_BRONZE_FILENAMES = []

_creation_col = (
    PURVIEW_BRONZE_FILTER_DATE_COLUMN
    if PURVIEW_BRONZE_FILTER_DATE_COLUMN in bronze.columns
    else "creation_date"
)
if COPILOT_BRONZE_FILENAMES:
    _file_col = "FileName" if "FileName" in bronze.columns else "file_name"
    bronze = bronze.filter(F.col(_file_col).isin(COPILOT_BRONZE_FILENAMES))
bronze = filter_through_close_month_ts(bronze, _creation_col, _CLOSE)


In [ ]:
apply_local_merge_spark_conf(spark)

min_date = purview_copilot_min_creation_date(_CLOSE)
print(f"purview_copilot_usage: creation_date >= {min_date}", flush=True)

write_df, _copilot_okta_rejects = build_purview_copilot_usage_silver(
    bronze,
    okta,
    hc,
    min_creation_date=min_date,
    close_month_no=_CLOSE,
    frozen_month_nos=_FROZEN,
)
_copilot_batch_months = _distinct_month_nos(
    write_df, "purview_copilot_usage_month_no"
)
publish_merge_staging(write_df, TABLE_PURVIEW_COPILOT_SILVER)
if _copilot_okta_rejects.head(1):
    publish_merge_staging(
        _copilot_okta_rejects, TABLE_PURVIEW_COPILOT_OKTA_REJECT
    )


In [ ]:
_silver_staging = merge_staging_path(TABLE_PURVIEW_COPILOT_SILVER)
write_df = spark.read.parquet(str(_silver_staging))
_reject_staging = merge_staging_path(TABLE_PURVIEW_COPILOT_OKTA_REJECT)
if _reject_staging.is_dir() and any(_reject_staging.glob("*.parquet")):
    _rej = spark.read.parquet(str(_reject_staging))
else:
    _rej = spark.createDataFrame(
        [],
        "purview_copilot_usage_record_id string, purview_copilot_usage_month_no int, okta_reject_reason string, okta_reject_loaded_at timestamp",
    )
write_copilot_okta_rejects(spark, _rej, _copilot_batch_months)

if copilot_close_month_no() is not None:
    write_df = replace_one_month_in_frame(
        spark,
        TABLE_PURVIEW_COPILOT_SILVER,
        write_df,
        "purview_copilot_usage_month_no",
        int(copilot_close_month_no()),
    )
    publish_merge_staging(write_df, TABLE_PURVIEW_COPILOT_SILVER)
    write_full_table(write_df, TABLE_PURVIEW_COPILOT_SILVER)
else:
    write_copilot_silver_daily(
        spark,
        write_df,
        TABLE_PURVIEW_COPILOT_SILVER,
        PURVIEW_COPILOT_MERGE_KEY,
        close_month_no=copilot_close_month_no(),
    )
